# Text Clustering — K-Means on TF-IDF Vectors

**Goal:** Group sentences into clusters by topic — without any labels (unsupervised learning).

---

## How It Works

### Step 1 — TF-IDF Vectorization
Each sentence becomes a numeric vector where each dimension is a word's TF-IDF score.

### Step 2 — K-Means Clustering

1. Randomly place `k` centroids in vector space
2. Assign each sentence to the nearest centroid
3. Move each centroid to the mean of its assigned sentences
4. Repeat steps 2–3 until stable

```
Vector space (simplified 2D):

  python ●    ← cluster A (tech)
  machine ●
              football ●    ← cluster B (sports)
              cricket  ●
```

**K-Means does NOT know what the clusters mean** — it only groups by similarity.  
You interpret the clusters after the fact.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import pandas as pd

# ── Corpus ─────────────────────────────────────────────────
sentences = [
    "python programming language",
    "machine learning models",
    "data science analysis",
    "football match today",
    "cricket players practice",
]

# ── Step 1: Vectorize ──────────────────────────────────────
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(sentences)

print("Vocabulary:", vectorizer.get_feature_names_out())
print()
print("TF-IDF Matrix shape:", X.shape, "(5 sentences × vocab size)")


Vocabulary: ['analysis' 'cricket' 'data' 'football' 'language' 'learning' 'machine'
 'match' 'models' 'players' 'practice' 'programming' 'python' 'science'
 'today']

TF-IDF Matrix shape: (5, 15) (5 sentences × vocab size)


In [2]:
# ── Step 2: Cluster ───────────────────────────────────────
N_CLUSTERS = 2
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

# ── Step 3: Display results ────────────────────────────────
results = pd.DataFrame({"Sentence": sentences, "Cluster": labels})
print("Clustering Results:")
print(results.to_string(index=False))

print()
print("Sentences per cluster:")
for c in range(N_CLUSTERS):
    group = results[results["Cluster"] == c]["Sentence"].tolist()
    print(f"  Cluster {c}: {group}")


Clustering Results:
                   Sentence  Cluster
python programming language        0
    machine learning models        0
      data science analysis        0
       football match today        0
   cricket players practice        1

Sentences per cluster:
  Cluster 0: ['python programming language', 'machine learning models', 'data science analysis', 'football match today']
  Cluster 1: ['cricket players practice']


In [3]:
# ── Step 4: Inspect cluster centroids ─────────────────────
# The centroid shows which words define each cluster
feature_names = vectorizer.get_feature_names_out()

print("Top words per cluster centroid:")
for i, centroid in enumerate(kmeans.cluster_centers_):
    top_indices = centroid.argsort()[-3:][::-1]   # top 3 words
    top_words   = [feature_names[j] for j in top_indices]
    print(f"  Cluster {i}: {top_words}")


Top words per cluster centroid:
  Cluster 0: ['today', 'science', 'python']
  Cluster 1: ['practice', 'players', 'cricket']


## Key Concepts

| Concept | Meaning |
|---------|---------|
| Unsupervised | No labels — the model finds structure on its own |
| K-Means | Groups data into K clusters by minimizing within-cluster distance |
| Centroid | The average vector of all points in a cluster |
| `n_init=10` | Run K-Means 10 times with different seeds, pick the best |

## Limitations of This Example
- Small corpus → clusters may not be perfectly separated
- K-Means assumes **spherical clusters** — works poorly on complex shapes
- You must choose `k` upfront (try the **elbow method** to pick `k`)

## What to Try Next
- Add 2 more sports sentences and 2 more tech sentences — do clusters improve?
- Try `k=3` — what does the third cluster contain?
- Use `sklearn.metrics.silhouette_score` to measure cluster quality
- Visualize clusters with **PCA** (reduce to 2D) + `matplotlib`